In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/pre_cell_3.pickle

trying: ['lineitem']


me:  1
trying: ['pd']
me:  0
trying: ['load_customer']
me:  3
trying: ['customer']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['orders']


me:  2


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Define the cutoff date as a pandas Timestamp (will be handled on GPU)
date = pd.Timestamp("1995-03-04")

# 1) Filter rows and select only the needed columns, all on GPU
fcustomer = customer.loc[
    customer.C_MKTSEGMENT == "HOUSEHOLD",
    ["C_CUSTKEY"]
]

forders = orders.loc[
    orders.O_ORDERDATE < date,
    ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]
]

flineitem = lineitem.loc[
    lineitem.L_SHIPDATE > date,
    ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
]

# 2) Perform the two joins entirely on GPU
jn2 = (
    fcustomer
      .merge(forders,    left_on="C_CUSTKEY", right_on="O_CUSTKEY")
      .merge(flineitem,  left_on="O_ORDERKEY", right_on="L_ORDERKEY")
)

# 3) Compute the TMP column, group, sum and sort—all GPU operations
jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)

res = (
    jn2
      .groupby(
          ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
          as_index=False,
          sort=False
      )["TMP"]
      .sum()
      .sort_values("TMP", ascending=False)
)

# res now has the same schema [L_ORDERKEY, TMP, O_ORDERDATE, O_SHIPPRIORITY]


CPU times: user 144 ms, sys: 44.2 ms, total: 189 ms
Wall time: 197 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_2.pickle

migration speed (bps): 812819719.3863275
---------------------------
variables to migrate:
forders 48
jn2 48
load_lineitem 144
lineitem 48
pd 72
res 48
customer 48
STORAGE_OPTS 64
load_orders 144
date 48
tpch_parent 54
orders 48
load_customer 144
flineitem 48
fcustomer 48
DATA_ROOT 80
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables ['lineitem', 'customer', 'orders']
Active variables []
Intermediate variables ['fcustomer', 'date', 'jn2', 'forders', 'flineitem', 'res']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q03/opt_cell_exec_info_3_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
res_opt = res
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
